# Pipeline Demo — Phase 5

Demonstrates the new `portfolio.pipeline` module that replaces the
~80 lines of boilerplate duplicated across cost_analysis, validation,
risk_analysis, and alpha_iteration notebooks.

**Pipeline stages:**
1. `compute_daily_returns()` — OHLCV → daily return series
2. `compute_next_day_returns()` — daily returns → tradable next-day returns
3. `build_factor_pipeline()` — OHLCV → preprocessed composite factor
4. `build_sizing_methods()` — composite → 4 sizing method weights
5. `resample_weights()` — reduce rebalancing frequency
6. `compute_portfolio_return()` — weights × returns → portfolio returns
7. `run_alpha_pipeline()` — all-in-one wrapper

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath("../src"))

from data.universe import get_universe, list_universes, US_LARGE_CAP_50
from portfolio.pipeline import (
    compute_daily_returns,
    compute_next_day_returns,
    compute_portfolio_return,
    resample_weights,
    build_factor_pipeline,
    build_sizing_methods,
    run_alpha_pipeline,
    list_factors,
)
from data.loader.data_loader import stock_load_process
import numpy as np

print(f"Available factors: {list_factors()}")
print(f"Available universes: {list_universes()}")
print(f"Default universe ({len(US_LARGE_CAP_50)} tickers): {US_LARGE_CAP_50[:5]}...")

Available factors: ['bbiboll', 'momentum', 'vol_ratio']
Available universes: ['US_LARGE_CAP_50', 'US_LARGE_CAP_52']
Default universe (51 tickers): ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'NVDA']...


## Load OHLCV Data

In [2]:
tickers = get_universe("US_LARGE_CAP_50")
ohlcv = stock_load_process(
    tickers=tickers,
    start_date="2023-06-01",
    end_date="2025-12-31",
).collect()
print(f"OHLCV shape: {ohlcv.shape}")
ohlcv.head(3)

Loading from cache: /mnt/blackdisk/quant_data/polygon_data/processed/us_stocks_sip/day_aggs_v1/cache_536cbb0024dad6f108df7a61e803f2fd.parquet
Cache loaded: 33,175 rows, 2.13 MB
OHLCV shape: (33175, 10)


ticker,timestamps,volume,open,close,high,low,window_start,transactions,split_date
str,"datetime[ns, America/New_York]",i64,f64,f64,f64,f64,i64,u32,date
"""AAPL""",2023-06-01 00:00:00 EDT,66635929,177.7,180.09,180.12,176.9306,1685592000000000000,624079,null
"""AAPL""",2023-06-02 00:00:00 EDT,60974501,181.03,180.95,181.78,179.26,1685678400000000000,617186,null
"""AAPL""",2023-06-05 00:00:00 EDT,120530432,182.63,179.58,184.951,178.035,1685937600000000000,1286123,null


## Stage 1-2: Daily & Next-Day Returns

In [3]:
daily_returns = compute_daily_returns(ohlcv)
next_day_returns = compute_next_day_returns(daily_returns)

print(f"Daily returns: {daily_returns.shape}")
print(f"Next-day returns: {next_day_returns.shape}")
daily_returns.head(5)

Daily returns: (33123, 3)
Next-day returns: (33071, 3)


date,ticker,daily_return
"datetime[ns, America/New_York]",str,f64
2023-06-02 00:00:00 EDT,"""AAPL""",0.004775
2023-06-05 00:00:00 EDT,"""AAPL""",-0.007571
2023-06-06 00:00:00 EDT,"""AAPL""",-0.00206
2023-06-07 00:00:00 EDT,"""AAPL""",-0.007756
2023-06-08 00:00:00 EDT,"""AAPL""",0.015465


## Stage 3: Factor Pipeline

In [4]:
composite = build_factor_pipeline(
    ohlcv,
    factor_names=["bbiboll", "vol_ratio"],
    combination_method="equal_weight",
)
print(f"Composite factor shape: {composite.shape}")
composite.head(5)

Composite factor shape: (31459, 3)


date,ticker,value
"datetime[ns, America/New_York]",str,f64
2023-07-20 00:00:00 EDT,"""AAPL""",-0.147162
2023-07-21 00:00:00 EDT,"""AAPL""",-0.141948
2023-07-24 00:00:00 EDT,"""AAPL""",-0.5581
2023-07-25 00:00:00 EDT,"""AAPL""",-0.144108
2023-07-26 00:00:00 EDT,"""AAPL""",-0.151351


## Stage 4-5: Sizing & Rebalancing

In [5]:
sizing = build_sizing_methods(composite, ohlcv, daily_returns)

for name, w in sizing.items():
    resampled = resample_weights(w, rebal_every_n=5)
    print(f"{name:25s}  raw={w.shape}  resampled={resampled.shape}")

Equal-Weight               raw=(12320, 3)  resampled=(12320, 3)
Inverse-Vol                raw=(12320, 3)  resampled=(12320, 3)
Vol-Target (10%)           raw=(12320, 3)  resampled=(12320, 3)
Half-Kelly                 raw=(30055, 3)  resampled=(30053, 3)


## Stage 6: Portfolio Returns per Sizing Method

In [6]:
for name, w in sizing.items():
    w_resampled = resample_weights(w, rebal_every_n=5)
    port_ret = compute_portfolio_return(w_resampled, next_day_returns)
    arr = port_ret["port_return"].to_numpy()
    sharpe = np.mean(arr) / np.std(arr, ddof=1) * np.sqrt(252)
    print(f"{name:25s}  Sharpe={sharpe:+.3f}  n_days={len(arr)}")

Equal-Weight               Sharpe=-0.443  n_days=615
Inverse-Vol                Sharpe=-0.591  n_days=615
Vol-Target (10%)           Sharpe=-0.450  n_days=615
Half-Kelly                 Sharpe=+0.790  n_days=588


## All-in-One: `run_alpha_pipeline()`

Single function call replaces all the boilerplate above.

In [7]:
result = run_alpha_pipeline(
    ohlcv,
    factor_names=["bbiboll", "vol_ratio"],
    sizing_method="Half-Kelly",
    rebal_every_n=5,
    n_long=10,
    n_short=10,
)

print(f"Result keys: {sorted(result.keys())}")
print(f"\nSharpe:        {result['sharpe']:+.3f}")
print(f"Annual Return: {result['annual_return']:+.2%}")
print(f"Annual Vol:    {result['annual_vol']:.2%}")
print(f"Trading days:  {result['n_days']}")

Result keys: ['annual_return', 'annual_vol', 'composite', 'daily_returns', 'n_days', 'next_day_returns', 'portfolio_returns', 'returns_array', 'sharpe', 'sizing_methods', 'weights']

Sharpe:        +0.790
Annual Return: +3.87%
Annual Vol:    4.89%
Trading days:  588
